# LOAD FILE VÀ RUN CÁC CELL BÊN DƯỚI

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 56.3 MB/s eta 0:00:00


In [4]:
import fitz
import re
import json

pdf_path = "nghi_dinh_168.pdf"
doc = fitz.open(pdf_path)

# =========================
# 1. LOAD TEXT
# =========================
text = ""
for page in doc:
    text += page.get_text() + "\n"

lines = [l.strip() for l in text.split("\n") if l.strip()]

# =========================
# 2. PATTERNS
# =========================
re_chuong = re.compile(r"^Chương\s+[IVXLC\d]+", re.IGNORECASE)
re_dieu = re.compile(r"^Điều\s+(\d+)\.")
re_khoan = re.compile(r"^(\d+)\.")

# =========================
# 3. DATA
# =========================
data = []

current_chuong = None
current_dieu = None
current_khoan = None

# =========================
# 4. PARSE
# =========================
for line in lines:

    # ===== CHƯƠNG =====
    if re_chuong.match(line):
        if current_khoan:
            current_dieu["khoan"].append(current_khoan)
        if current_dieu:
            current_chuong["dieu"].append(current_dieu)
        if current_chuong:
            data.append(current_chuong)

        current_chuong = {"chuong": line, "dieu": []}
        current_dieu = None
        current_khoan = None
        continue

    # ===== ĐIỀU =====
    match_dieu = re_dieu.match(line)
    if match_dieu:
        if current_khoan:
            current_dieu["khoan"].append(current_khoan)
        if current_dieu:
            current_chuong["dieu"].append(current_dieu)

        # Lưu thêm dieu_number để làm metadata cho RAG sau này
        current_dieu = {
            "dieu_number": match_dieu.group(1),
            "title": line,
            "khoan": []
        }
        current_khoan = None
        continue

    # ===== KHOẢN =====
    match_khoan = re_khoan.match(line)
    if match_khoan:
        if current_khoan:
            current_dieu["khoan"].append(current_khoan)

        # SỬA LỖI Ở ĐÂY:
        # Không dùng 'title' cho Khoản nữa. Gán luôn dòng đầu tiên vào 'content'.
        # Trích xuất số của Khoản để lưu trữ.
        current_khoan = {
            "khoan_number": match_khoan.group(1),
            "content": line
        }
        continue

    # ===== CONTINUE TEXT =====
    # Xử lý nối các dòng bị ngắt vật lý trong PDF
    if current_khoan:
        current_khoan["content"] += " " + line
    elif current_dieu:
        # Nếu tiêu đề Điều dài và bị xuống dòng
        current_dieu["title"] += " " + line
    elif current_chuong:
        # Nếu tên Chương dài và bị xuống dòng
        current_chuong["chuong"] += " " + line

# =========================
# 5. PUSH LAST
# =========================
if current_khoan:
    current_dieu["khoan"].append(current_khoan)

if current_dieu:
    current_chuong["dieu"].append(current_dieu)

if current_chuong:
    data.append(current_chuong)

# =========================
# 6. SAVE
# =========================
with open("law_structured.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("DONE CLEAN (NO DIEM)")

DONE CLEAN (NO DIEM)


In [6]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemetry-proto-1.38.0:
      Successfully uninstalled opentelemetry-proto-1.38.0
  Attempting uninstall: opentelemetry-exporter-otlp-proto-common
    Found existing installation: opentelemetry-exporter-otlp-proto-common 1.38.0
    Uninstalling opentelemetry-exporter-otlp-proto-common-1.38.0:
      Successfully uninstalled opentelemetry-exporter-otlp-proto-common-1.38.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uni

In [15]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. INIT MODEL + DB
# ==========================================
print("🚀 Initializing...")

model = SentenceTransformer('keepitreal/vietnamese-sbert')

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="nghi_dinh_168")


# ==========================================
# 2. LOAD DATA + BUILD TEXTS
# ==========================================
print("📂 Loading JSON...")

with open("law_structured.json", "r", encoding="utf-8") as f:
    data = json.load(f)

documents = []
texts_to_embed = []


for chuong in data:
    chuong_name = chuong.get("chuong", "")

    for dieu in chuong.get("dieu", []):
        dieu_number = dieu.get("dieu_number", "")

        for khoan in dieu.get("khoan", []):
            khoan_number = khoan.get("khoan_number", "")
            content = khoan.get("content", "")

            # =========================
            # ENRICHED CONTEXT TEXT
            # =========================
            text = f"""
Chương: {chuong_name}
Điều: {dieu_number}
Khoản: {khoan_number}

Nội dung:
{content}
""".strip()

            texts_to_embed.append(text)

            doc_id = f"dieu_{dieu_number}_khoan_{khoan_number}"

            documents.append({
                "id": doc_id,
                "text": text,
                "metadata": {
                    "nguon": "Nghị định 168",
                    "chuong": chuong_name,
                    "dieu_number": dieu_number,
                    "khoan_number": khoan_number
                }
            })

print(f"📊 Total chunks: {len(texts_to_embed)}")


# ==========================================
# 3. EMBEDDING
# ==========================================
print("🧠 Embedding...")

embeddings = model.encode(texts_to_embed, show_progress_bar=True)


# ==========================================
# 4. DEDUP + UPSERT
# ==========================================
print("💾 Saving to ChromaDB...")

unique_docs = {}

for i, doc in enumerate(documents):
    doc["vector"] = embeddings[i].tolist()
    unique_docs[doc["id"]] = doc

final_docs = list(unique_docs.values())

ids = [d["id"] for d in final_docs]
vectors = [d["vector"] for d in final_docs]
metadatas = [d["metadata"] for d in final_docs]
texts = [d["text"] for d in final_docs]

collection.upsert(
    ids=ids,
    embeddings=vectors,
    metadatas=metadatas,
    documents=texts
)

print("✅ Done indexing!")

🚀 Initializing...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📂 Loading JSON...
📊 Total chunks: 332
🧠 Embedding...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

💾 Saving to ChromaDB...
✅ Done indexing!


In [17]:
def search_law(query, n_results=3):
    query_vector = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_vector,
        n_results=n_results
    )

    for i in range(len(results['documents'][0])):
        print(f"Top {i+1}:")
        print(f"Nội dung: {results['documents'][0][i]}")
        print(f"Metadata: {results['metadatas'][0][i]}")
        print("-" * 30)

search_law("Hành vi điều khiển xe máy dừng xe, đỗ xe ở lòng đường gây cản trở giao thông bị xử phạt như thế nào?")

Top 1:
Nội dung: Chương: Chương II HÀNH VI VI PHẠM, HÌNH THỨC, MỨC XỬ PHẠT, MỨC TRỪ ĐIỂM GIẤY PHÉP LÁI XE VÀ BIỆN PHÁP KHẮC PHỤC HẬU QUẢ VI PHẠM HÀNH CHÍNH VỀ TRẬT TỰ, AN TOÀN GIAO THÔNG TRONG LĨNH VỰC GIAO THÔNG ĐƯỜNG BỘ Mục 1. VI PHẠM QUY TẮC GIAO THÔNG ĐƯỜNG BỘ
Điều: 8
Khoản: 2

Nội dung:
2. Phạt tiền từ 600.000 đồng đến 800.000 đồng đối với người điều khiển xe thực hiện một trong các hành vi vi phạm sau đây: a) Dừng xe, đỗ xe trên phần đường xe chạy ở đoạn đường ngoài đô thị nơi có lề đường rộng; dừng xe, đỗ xe không sát mép đường phía bên phải theo chiều đi ở nơi đường có lề đường hẹp hoặc không có lề đường; dừng xe, đỗ xe ngược với chiều lưu thông của làn đường; dừng xe, đỗ xe trên dải phân cách cố định ở giữa hai phần đường xe chạy; dừng xe, đỗ xe không đúng vị trí quy định ở những đoạn đường đã có bố trí nơi dừng xe, đỗ xe; đỗ xe trên dốc không chèn bánh; dừng xe nơi có biển “Cấm dừng xe và đỗ xe”; đỗ xe nơi có biển “Cấm đỗ xe” hoặc biển “Cấm dừng xe và đỗ xe”, trừ hành vi vi p

In [9]:
# =========================
# INSTALL (run once)
# =========================
!pip install fastapi uvicorn pyngrok sentence-transformers chromadb transformers torch pandas


# KHỞI ĐỘNG MODEL
Vào trang ngrok đăng ký tài khoản và copy cái TOKEN nó cho vào ô có chữ "NÉM TOKEN VÀO ĐÂY". Ném xong rồi run, copy cái đường dẫn nó cung cấp (đọc file readme)


In [ ]:
# =========================
# IMPORTS
# =========================
from flask import Flask, request, jsonify
from pyngrok import ngrok

import torch
import chromadb
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# =========================
# NGROK
# =========================
ngrok.set_auth_token("NÉM TOKEN VÀO ĐÂY")

# =========================
# FLASK
# =========================
app = Flask(__name__)

# =========================
# GLOBALS
# =========================
embed_model = None
collection = None
tokenizer = None
llm_model = None
term_dict = None

doc_texts = []


# =========================
# LOAD MAPPING
# =========================
def load_mapping(file_path):
    df = pd.read_csv(file_path, sep=';')
    df.columns = df.columns.str.strip()

    term_dict = {}

    for _, row in df.iterrows():
        formal = str(row[df.columns[0]]).strip()
        casual_list = str(row[df.columns[1]]).split(',')

        for c in casual_list:
            c = c.strip().lower()
            if c:
                term_dict[c] = formal

    print("✅ Mapping loaded:", len(term_dict))
    return term_dict


# =========================
# INIT SYSTEM
# =========================
def initialize_system():
    global embed_model, collection, tokenizer, llm_model, term_dict, doc_texts

    print("🚀 Loading system...")

    embed_model = SentenceTransformer("keepitreal/vietnamese-sbert")

    chroma_client = chromadb.PersistentClient(path="./chroma_db")
    collection = chroma_client.get_collection(name="nghi_dinh_168")

    model_name = "Qwen/Qwen2.5-3B-Instruct"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    llm_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    term_dict = load_mapping("./mapping.csv")

    # load all docs (for rerank stage)
    data = collection.get()
    doc_texts = data["documents"]

    print("✅ SYSTEM READY!")


# =========================
# QUERY MAPPING (formalization)
# =========================
def map_query(query):
    q = query.lower()

    for casual, formal in term_dict.items():
        if casual in q:
            q = q.replace(casual, formal)

    return q


# =========================
# STAGE 1: RETRIEVE CANDIDATES
# =========================
def retrieve_candidates(query, top_k=30):

    refined_query = map_query(query)

    emb = embed_model.encode(refined_query, normalize_embeddings=True)

    results = collection.query(
        query_embeddings=[emb.tolist()],
        n_results=top_k
    )

    docs = results.get("documents", [[]])[0]

    return docs


# =========================
# STAGE 2: RERANK
# =========================
def rerank(query, docs, top_k=5):

    query_emb = embed_model.encode(query, normalize_embeddings=True)

    scored = []

    for doc in docs:
        doc_emb = embed_model.encode(doc, normalize_embeddings=True)

        score = np.dot(query_emb, doc_emb)  # cosine similarity

        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [d for _, d in scored[:top_k]]


# =========================
# FULL RETRIEVAL PIPELINE
# =========================
def retrieve(query):

    # Stage 1
    candidates = retrieve_candidates(query, top_k=30)

    # Stage 2
    top_docs = rerank(query, candidates, top_k=5)

    return "\n".join(top_docs)


# =========================
# GENERATION
# =========================
def generate_answer(query, context):

    if not context:
        return "Không tìm thấy dữ liệu"

    prompt = f"""
Dữ liệu pháp luật NĐ168 năm 2024:
{context}

Câu hỏi: {query}

Yêu cầu:
- Trả lời chính xác theo luật
- Trích điều/khoản nếu có
"""

    messages = [
        {"role": "system", "content": "Bạn là chuyên gia pháp lý Việt Nam."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,        # 🔥 QUAN TRỌNG NHẤT
            temperature=0.0,        # deterministic
            top_p=1.0,
            repetition_penalty=1.1
        )

    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )


# =========================
# PIPELINE
# =========================
def ask(query):
    context = retrieve(query)
    answer = generate_answer(query, context)
    return answer, context


# =========================
# API
# =========================
@app.route("/ask", methods=["POST"])
def ask_api():
    data = request.get_json()

    if "question" not in data:
        return jsonify({"error": "missing question"}), 400

    answer, context = ask(data["question"])

    return jsonify({
        "question": data["question"],
        "answer": answer,
        "context": context
    })


# =========================
# RUN SERVER
# =========================
if __name__ == "__main__":
    initialize_system()

    public_url = ngrok.connect(8000)
    print("🔥 NGROK:", public_url)

    app.run(host="0.0.0.0", port=8000)

🚀 Loading system...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Mapping loaded: 533
✅ SYSTEM READY!
🔥 NGROK: NgrokTunnel: "https://exciting-depraved-hardness.ngrok-free.dev" -> "http://localhost:8000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [05/May/2026 09:00:01] "POST /ask HTTP/1.1" 200 -
